# Assignment 3: Neural Network for Avocado Price Prediction
## Data Preprocessing & Neural Network Model

In [3]:
# Section 1: Load and Explore Data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('c:\\Users\\LENOVO\\Downloads\\avocado.csv')

# Display basic info
print("Dataset shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# Section 2: Data Preprocessing
# Drop unnecessary columns
df = df.drop(['Unnamed: 0', 'Date'], axis=1)

# Encode categorical variable 'type' (organic/conventional)
df['type_encoded'] = (df['type'] == 'organic').astype(int)
df = df.drop('type', axis=1)

# Encode 'region' using one-hot encoding
region_dummies = pd.get_dummies(df['region'], prefix='region', drop_first=True)
df = pd.concat([df, region_dummies], axis=1)
df = df.drop('region', axis=1)

print("Preprocessed columns:", df.columns.tolist())
print("Data shape after preprocessing:", df.shape)

In [ ]:
# Section 3: Feature Engineering
# Target variable is AveragePrice
y = df['AveragePrice'].values

# Features: all columns except target
X = df.drop('AveragePrice', axis=1).values

# Normalize features using StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Feature matrix shape:", X_scaled.shape)
print("Target variable shape:", y.shape)
print("Features normalized (mean ≈ 0, std ≈ 1)")

In [ ]:
# Section 4: Train-Test Split
# Split data: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")
print(f"Feature dimensions: {X_train.shape[1]}")

In [ ]:
# Section 5: Build Neural Network Model
model = keras.Sequential([
    # Input layer with feature dimension
    layers.Dense(64, activation='relu', input_dim=X_train.shape[1]),
    
    # Hidden layer 1: 64 neurons with ReLU activation
    layers.Dense(64, activation='relu'),
    
    # Hidden layer 2: 32 neurons with ReLU activation
    layers.Dense(32, activation='relu'),
    
    # Output layer: 1 neuron for regression (price prediction)
    layers.Dense(1)
])

print("Model Architecture:")
model.summary()

In [ ]:
# Section 6: Compile and Train the Model
# Compile: optimizer=Adam, loss=MSE (Mean Squared Error)
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Train the model for 50 epochs
history = model.fit(X_train, y_train, 
                    epochs=50, 
                    batch_size=32,
                    validation_split=0.2,  # 20% for validation
                    verbose=0)

print("Model training completed!")

In [ ]:
# Section 7: Evaluate Model Performance
# Make predictions on test set
y_pred = model.predict(X_test, verbose=0)

# Calculate error metrics
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)

print(f"Test Set Performance:")
print(f"  RMSE: ${rmse:.2f}")
print(f"  MAE:  ${mae:.2f}")

# Plot training and validation loss
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.title('Model Training History')
plt.grid(True)
plt.show()

# Display sample predictions vs actual
print("\nSample Predictions vs Actual:")
for i in range(5):
    print(f"Predicted: ${y_pred[i][0]:.2f}, Actual: ${y_test[i]:.2f}")